# Lab 4: Parameter-efficient fine-tuning

Fine-tuning all parameters of pre-trained language models can be resource-intensive. Because of this, current research in natural language processing is looking into developing methods for adapting models to downstream tasks without full fine-tuning. These methods only tune a small number of model parameters while yielding performance comparable to that of a fully fine-tuned model.

In this lab, you will implement LoRA, one of the most well-known methods for parameter-efficient fine-tuning. LoRA stands for “Low-Rank Adaptation of Large Language Models” and was originally described in a research article by [Hu et al. (2021)](https://arxiv.org/abs/2106.09685).

Along the way, you will earn experience with [Hugging Face Transformers](https://huggingface.co/docs/transformers/en/index), a state-of-the-art library for training and deploying language models, as well as with several related libraries. In particular, you will learn a best-practice workflow for downloading a Transformer model and fine-tuning it on the downstream task of binary sentiment classification.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

## Dataset

The data for this lab comes from the [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/). The full dataset consists of 50,000 highly polar movie reviews collected from the Internet Movie Database (IMDB). Here, we use a random sample consisting of 2,000 reviews for training and 500 reviews for evaluation.

To load the dataset, we use the [Hugging Face Datasets](https://huggingface.co/docs/datasets/en/index) library.

First, let's set up the Python path and working directory:

In [1]:
from datasets import load_dataset
from torch import nn 
from transformers import DistilBertForSequenceClassification
import torch
from typing import Any, Dict, List
import copy
import functools
imdb_dataset = load_dataset(
    "csv", data_files={"train": "train.csv", "eval": "eval.csv"}
)

imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'review', 'label'],
        num_rows: 2000
    })
    eval: Dataset({
        features: ['index', 'review', 'label'],
        num_rows: 500
    })
})

As we can see, each sample in the dataset is a record with three fields: an internal index (`index`, an integer), the text of the review (`review`, a string), and the sentiment label (`label`, an integer – 1&nbsp;for “positive” and 0&nbsp;for “negative” sentiment).

Here is an example record:

In [2]:
imdb_dataset["train"][645]

{'index': 2981,
 'review': 'Brilliant execution in displaying once and for all, this time in the venue of politics, of how "good intentions do actually pave the road to hell". Excellent!',
 'label': 1}

## Tokeniser

As our pre-trained language model, we will use [DistilBERT](https://huggingface.co/docs/transformers/en/model_doc/distilbert), a compact encoder model with 40% less parameters than BERT base. DistilBERT is not actually a *large* language model by modern standards and thus does not benefit as much from parameter-efficient fine-tuning as other models. However, it has the benefit of being light and fast, and can be run even on consumer hardware.

To feed the movie reviews to DistilBERT, we need to tokenise them and encode the resulting tokens as integers in the model vocabulary. We start by loading the DistilBERT tokeniser using the [Auto classes](https://huggingface.co/docs/transformers/en/model_doc/auto):

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

We then create a tokenised version of the dataset:

In [4]:
def tokenize_function(batch):
    return tokenizer(batch["review"], padding=True, truncation=True)


tokenized_imdb_dataset = imdb_dataset.map(tokenize_function, batched=True)

tokenized_imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'review', 'label', 'input_ids', 'attention_mask'],
        num_rows: 2000
    })
    eval: Dataset({
        features: ['index', 'review', 'label', 'input_ids', 'attention_mask'],
        num_rows: 500
    })
})

As we can see, tokenising adds two additional fields to each review: `input_ids` is the list of token ids corresponding to the review, and `attention_mask` is the list of indices specifying which tokens the encoder should attend to.

To avoid trouble when fine-tuning the model later, the next cell disables tokeniser parallelism.

In [5]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Trainer

In this section, we will set up our workflow for training and evaluating DistilBERT models. The central component in this workflow is the [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer), which provides extensive configuration options. Here, we leave most of these options at their default value. Two changes we *do* make are to enable evaluation of the trained model after each epoch, and to log the training and evaluation loss after every 5&nbsp;training steps (the default is 500).

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="tmp_trainer",
    eval_strategy="epoch",
    logging_steps=5,
)

In addition to the loss, we also track classification accuracy. For this we import the [Hugging Face Evaluate](https://huggingface.co/docs/evaluate/en/index) library and define a small helper function `compute_metrics()` that the trainer will call after each epoch.

In [7]:
import evaluate

accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In the next cell we define a convenience function `make_trainer()` that creates a readily-configured trainer for a specified model (*model*). We will use this trainer both to train the model on the training section of the tokenised review dataset, and to evaluate it on the evaluation section.

In [8]:
from transformers import Trainer


def make_trainer(model):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_imdb_dataset["train"],
        eval_dataset=tokenized_imdb_dataset["eval"],
        compute_metrics=compute_metrics,
    )
    return trainer

## Full fine-tuning

In the rest of this notebook, we will work our way to the implementation of LoRA, and compare LoRA to traditional fine-tuning methods. Our first point of reference is a fully fine-tuned DistilBERT model.

We start by loading the pre-trained model:

In [9]:
from transformers import AutoModelForSequenceClassification
pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


The architecture of DistilBERT is that of a standard Transformer encoder with an embedding layer (`embeddings`) followed by a stack of six Transformer blocks (`transformer`) and a feedforward network with two linear layers (`pre_classifier` and `classifier`) and a final dropout layer (`dropout`).

### 🎈 Task 4.01: Counting the number of trainable parameters

One relevant measure in the context of parameter-efficient fine-tuning is the number of parameters that need to be changed when training a model. Your first task in this lab is to write a function `num_trainable_parameters()` that calculates this number for a given model.

In [10]:
def num_trainable_parameters(model: AutoModelForSequenceClassification):
    n_params = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            n_params += param.numel()
    print(f"Number of trainable parameters: {n_params}")
    return n_params

The function should implement the following specification:

> **num_trainable_parameters** (*model*)
>
> Returns the number of float-valued trainable parameters in the specified *model* as an integer.

#### 👍 Hint

The term *parameter* can refer to either complete tensors or the individual elements of these tensors. For example, a linear layer created by `nn.Linear(3, 5)` has 2&nbsp;tensor-valued parameters (a weight matrix and a bias vector) and 20&nbsp;float-valued parameters (the elements of these tensors). To get the tensor-valued parameters of a model, you can use the [`parameters()`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.parameters) method. A parameter is *trainable* if it requires gradient.

#### 🤞 Test your code

To test your code, apply your function to the pre-trained model. The correct number of float-valued trainable parameters for this model is 66,955,010.

In [11]:
num_trainable_parameters(pretrained_model)

Number of trainable parameters: 66955010


66955010

### Fine-tuning

When we load the pre-trained model, the Hugging Face Transformers library warns us that the weights of the feedforward network have not yet been trained. To do so, in the next cell, we pass the pre-trained model to a trainer and initiate the fine-tuning process.

**⚠️ Please note that fine-tuning the model will take some time! ⚠️**

You can work on the other problems in this lab while you are waiting.

In [12]:
finetuned_trainer = make_trainer(pretrained_model)

finetuned_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.485600,0.385469,0.864000
2,0.006900,0.393039,0.906000
3,0.001800,0.443451,0.916000


TrainOutput(global_step=750, training_loss=0.2398320487315456, metrics={'train_runtime': 124.5589, 'train_samples_per_second': 48.17, 'train_steps_per_second': 6.021, 'total_flos': 794804391936000.0, 'train_loss': 0.2398320487315456, 'epoch': 3.0})

Because full fine-tuning is so resource-intensive, we save the fine-tuned model to disk:

In [13]:
finetuned_trainer.save_model("finetuned")

Later in this notebook, whenever you need the fully fine-tuned version of the model, you can load it as follows:

In [14]:
finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned")

### Convenience functions

Because we will repeat the steps we just took to fine-tune the pre-trained model several times in this notebook, we define two convenience functions:

In [15]:
def train(model):
    print("Number of trainable parameters:", num_trainable_parameters(model))
    trainer = make_trainer(model)
    trainer.train()
    return model

In [16]:
def evaluate(model):
    trainer = make_trainer(model)
    return trainer.evaluate()

## Tuning the final layers only

If full fine-tuning marks one end of the complexity spectrum, the other end is marked by only tuning the final layers of the transformer – the *head* of the model. In the case of DistilBERT, the head consists of the `pre_classifier` and `classifier` layers.

### 🎈 Task 4.02: Head-tuning

Implement the head-tuning strategy by coding the following function:

In [17]:
def make_headtuned_model(use_cuda: bool = True):
    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
    if use_cuda:
        model = model.to("cuda")
    for name, param in model.named_parameters():
        if "pre_classifier" not in name and "classifier" not in name:
            param.requires_grad = False
        else:
            param.requires_grad = True
    return train(model)

Here is the specification of this function:

> **make_headtuned_model** ()
>
> Returns a model that is identical to the pre-trained model, except that the head layers have been trained on the sentiment data. (The other parameters of the pre-trained model are left untouched.)

#### 👍 Hint

You freeze a parameter by setting its `requires_grad`-attribute to `False`.

Once you have an implementation of the head-tuning strategy, evaluate it on the evaluation data. How much accuracy do we lose when only training the final layers of the pre-trained model, compared to full fine-tuning?

In [18]:
headtuned_model = make_headtuned_model()
evaluate(headtuned_model)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Number of trainable parameters: 592130
Number of trainable parameters: 592130


Epoch,Training Loss,Validation Loss,Accuracy
1,0.586000,0.598972,0.712000
2,0.453000,0.539054,0.758000
3,0.529100,0.513854,0.812000


{'eval_loss': 0.5138542652130127,
 'eval_model_preparation_time': 0.0006,
 'eval_accuracy': 0.812,
 'eval_runtime': 2.9963,
 'eval_samples_per_second': 166.871,
 'eval_steps_per_second': 21.026}

#### 🤞 Test your code

If you configured your model correctly, `num_trainable_parameters()` should show 592,130 trainable parameters.

### Answer to Task 4.02

We lose approximately 9% accuracy when only training the final layers of the pre-trained model compared to full fine-tuning. The accuracy decreased from approximately 90% to approximately 81%.

For future reference, we also save the head-tuned model:

In [19]:
make_trainer(headtuned_model).save_model("headtuned")

## Layer surgery

LoRA works by “wrapping” frozen layers from the pre-trained Transformer model inside adapter modules. Conventionally, this wrapping is only applied to the linear layers that transform the queries and values in the self-attention mechanism. To implement the wrapping, we need functions to extract and replace layers in a model. Your task in this section is to code these functions.

### 🎓 Task 4.03: Extracting layers

Code a function that extracts the query and value linear layers from a DistilBERT model:

In [20]:
def deep_getattr(obj: Any, attr: str) -> Any:
    return functools.reduce(getattr, attr.split('.'), obj)

def deep_setattr(obj: Any, attr: str, value: Any) -> None:
    parts = attr.split('.')
    child = deep_getattr(obj, ".".join(parts[:-1]))
    setattr(child, parts[-1], value)

def extract(model: DistilBertForSequenceClassification) -> dict[str, nn.Module]:
    """Returns name and a linear layer for each query and value head in a distilbert
    transformer model
    """
    wrapping = {}
    for name, param in model.named_modules():
        if "attention.v_lin" in name or "attention.q_lin" in name:
            wrapping[name] = param
    for k,v in wrapping.items():
        assert isinstance(v, nn.Module)
    return copy.deepcopy(wrapping)
wrapping = extract(headtuned_model)

Implement this function to match the following specification:

> **extract** (*model*)
>
> Takes a DistilBERT model (*model*) and extracts the query and value linear layers from each block of the Transformer. Returns a dictionary mapping the DistilBERT module names of these layers to the layers themselves (instances of `nn.Linear`).

#### 👍 Hint

As we saw earlier, the DistilBERT model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. Use [`get_submodule()`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.get_submodule) to retrieve a layer by name. You can hard-wire the names of the layers you want to extract.

#### 🤞 Test your code

To test your code, check the number of trainable float-valued parameters in the extracted layers. This number should be 7,087,104.

In [21]:
def count_params(wrapping):
    ctr=0
    for k,v in wrapping.items():
        ctr += v.weight.numel()
        ctr += v.bias.numel()
    return ctr
count_params(extract(headtuned_model))

7087104

### 🎓 Task 4.04: Replacing layers

Next, code the inverse of the `extract()` function to replace selected layers of a module using a dictionary of named layers.

In [22]:
import torch.nn as nn


def clone_linear(original):
    out_features, in_features = original.weight.shape
    copy = nn.Linear(in_features, out_features)
    copy.load_state_dict(original.state_dict())
    return copy
    
def replace(model: DistilBertForSequenceClassification, named_modules: Dict[str, nn.Module]):
      for name, new_module in named_modules.items():
          deep_setattr(model, name, clone_linear(new_module))
      return model

Implement this function to match the following specification:

> **replace** (*model*, *named_layers*)
>
> Takes a DistilBERT model (*model*) and a dictionary in the format returned by `extract()` (*named_layers*) and injects the extracted layers into the model. More specifically, suppose that *named_layers* contains a key–value pair `(name, layer)`. Then the function replaces the submodule of *model* addressed by the fully-qualified string name `name` by the layer `layer`. Returns the modified model.

#### 👍 Hint

Use [`getattr()`](https://docs.python.org/3/library/functions.html#getattr) and [`setattr()`](https://docs.python.org/3/library/functions.html#setattr) to return or set the value of a named submodule.

#### 🤞 Test your code

To test your implementation, write code that (1)&nbsp;extracts the query and value linear layers from the fine-tuned model; (2)&nbsp;replaces these layers with clones with random weights; and (3)&nbsp;replaces these layers again with the original versions. Evaluating the modified model after step&nbsp;(2) should yield a near-random accuracy. Evaluating it again after step&nbsp;(3) should yield the original accuracy.

The following function should be helpful. It clones a linear layer, copying the weights and the bias from the original.

In [23]:
# (1) extracts the q,v linear layers from fine-tuned model
fine_tuned_linear_layers = extract(pretrained_model) 
# (2) replace with random weights
random_linear_layers = { k: nn.Linear(*v.weight.shape) for k,v in fine_tuned_linear_layers.items() }
random_linear_layers_fine_tuned_model = copy.deepcopy(finetuned_model)
random_linear_layers_fine_tuned_model = replace(random_linear_layers_fine_tuned_model, random_linear_layers)
# (3) replace the layers again with fine-tuned weights
full_circle_fine_tuned_model = copy.deepcopy(random_linear_layers_fine_tuned_model)
full_circle_fine_tuned_model = replace(full_circle_fine_tuned_model, fine_tuned_linear_layers) # (2)
for i1,i2 in zip(pretrained_model.named_modules(), random_linear_layers_fine_tuned_model.named_modules()):
    k1, m1 = i1
    k2, m2 = i2
    assert k1 == k2
    assert  m1 != m2

In [24]:
print(evaluate(random_linear_layers_fine_tuned_model))
print(evaluate(full_circle_fine_tuned_model))


{'eval_loss': 0.7183455228805542, 'eval_model_preparation_time': 0.0006, 'eval_accuracy': 0.474, 'eval_runtime': 2.9821, 'eval_samples_per_second': 167.669, 'eval_steps_per_second': 21.126}


{'eval_loss': 0.4434509873390198, 'eval_model_preparation_time': 0.0006, 'eval_accuracy': 0.916, 'eval_runtime': 2.9901, 'eval_samples_per_second': 167.218, 'eval_steps_per_second': 21.07}


## Low-rank approximation

The basic idea behind LoRA is to conceptualise fine-tuned weights as a sum $W_0 + \Delta W$ of the weights from the pre-trained model, $W_0$, and a low-rank update matrix $\Delta W$. The goal of fine-tuning, then, is to learn the update matrix; this happens in the adapter layers.

Before we get to the implementation of the LoRA adapter layers, we first check to what extent the assumption that fine-tuning can be described by low-rank matrices holds true for DistilBERT. To do so, we will “cheat” and replace the query and value linear layers of the head-tuned model with low-rank approximations. The technical key to this is the truncated singular value decomposition (SVD).

### 🎓 Task 4.05: Low-rank matrix approximation

Your first task in this section is to implement the low-rank matrix approximation.

In [25]:
def approximate(matrix: torch.Tensor, rank: int) -> torch.Tensor:
    U, S, V = torch.svd_lowrank(matrix, q=rank)
    return  U @ torch.diag(S) @ V.T

Implement this function to match the following specification:

> **approximate** (*matrix*, *rank*)
>
> Takes a 2D-tensor (*matrix*) and an integer rank $r$ (*rank*), computes the truncated SVD with rank $r$ on the tensor, and returns the corresponding low-rank approximation matrix.

#### NOTE: 4.05 SVD RANKS
- "*Task 4.05: Just a heads up that I may ask more detailed questions about the SVD in the oral exam. For example, what are the ranks of the matrices U, S, V?*"
-    U      diag(q)      V^T   
- (m * q) @ (q * q) @ (q * n) = (m * n)

In [40]:
m = 1000
n = 1050
A = torch.rand(m,n)
q = 8

U, S, V = torch.svd_lowrank(A,q)
print("### RANKS OF SVD Decomp")
print(f"rank(U)={torch.linalg.matrix_rank(U)}, q={q}")
print(f"rank(S)={torch.linalg.matrix_rank(torch.diag(S))}, q={q}")
print(f"rank(V)={torch.linalg.matrix_rank(U)}, q={q}")




### RANKS OF SVD Decomp
rank(U)=8, q=8
rank(S)=8, q=8
rank(V)=8, q=8


#### 👍 Hint

If you need a refresher on the low-rank matrix approximation, read the corresponding section from the Wikipedia article on the [Singular value decomposition](https://en.wikipedia.org/wiki/Singular_value_decomposition#Low-rank_matrix_approximation). The truncated SVD is an extension of the full SVD; the latter can be computed using [`torch.linalg.svd()`](https://pytorch.org/docs/stable/generated/torch.linalg.svd.html).

#### 🤞 Test your code

To test your code, run the following cell. It creates a matrix `original` with rank $r \leq 8$ and after that the rank-$8$ approximation matrix `approximation`. You should find that the distance between the two matrices is very low.

In [26]:
original = torch.rand(768, 8) @ torch.rand(8, 384)
approximation = approximate(original, 8)
torch.dist(original, approximation)

tensor(0.0003)

### 🎓 Task 4.06: Approximated fine-tuned model (version 1)

In the next step, your task is to construct a version of the head-tuned model in which every query and value linear layer is replaced by a low-rank approximation of the corresponding layer from the fully fine-tuned model.

In [27]:
def make_approximated_model_1(rank) -> nn.Module:
    _headtuned_model = copy.deepcopy(headtuned_model)
    headtuned_qv_layers = extract(_headtuned_model) 
    for _, submodule in headtuned_qv_layers.items():
        approximate_layer: torch.Tensor = approximate(submodule.weight, rank)
        submodule.weight = nn.Parameter(approximate_layer)
    replace(_headtuned_model, headtuned_qv_layers)
    return _headtuned_model

Here is the specification of this function:

> **make_approximated_model_1** (*rank*)
>
> Takes an integer rank $r$ (*rank*) and returns a version of the head-tuned model in which every query and value linear layer is replaced by its $r$-approximated corresponding layer from the fully fine-tuned model.

Run the next cell to evaluate your model for different rank values. Start with the full rank and then halve the rank in each step. What is the lowest rank that still gives you a higher accuracy than the head-tuned model?

In [28]:
approximated_model_1 = make_approximated_model_1(768)

evaluate(approximated_model_1)

{'eval_loss': 0.5138283967971802,
 'eval_model_preparation_time': 0.0006,
 'eval_accuracy': 0.812,
 'eval_runtime': 2.9815,
 'eval_samples_per_second': 167.702,
 'eval_steps_per_second': 21.131}

### 🎓 Task 4.07: Approximated fine-tuned model (version 2)

In the approximated model from the previous section, the truncated SVD is applied to the full weight matrix of the fine-tuned model: $W_0 + \Delta W$. In LoRA, the low-rank approximation only applies to the *update matrix* $\Delta W$, i.e., the difference between the fully fine-tuned weights and the pre-trained weights.

In [41]:
def make_approximated_model_2(rank) -> nn.Module:
    _pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
    ).to("cuda")
    _finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned").to("cuda")
    _headtuned_model = AutoModelForSequenceClassification.from_pretrained("headtuned").to("cuda")
    pretrained_qv = extract(_pretrained_model)
    finetuned_qv = extract(_finetuned_model)
    headtuned_model_qv = extract(_headtuned_model)
    for name in pretrained_qv.keys():
        delta_W = finetuned_qv[name].weight.data - pretrained_qv[name].weight.data
        delta_W_approx = approximate(delta_W, rank)
        headtuned_model_qv[name].weight = nn.Parameter(
            pretrained_qv[name].weight.data + delta_W_approx
        )
    return replace(_headtuned_model, headtuned_model_qv)

Implement the function to match the following specification:

> **make_approximated_model_2** (*rank*)
>
> Takes an integer rank $r$ (*rank*) and returns a version of the head-tuned model in which the weight matrix of every query and value linear layer is replaced by the sum $W_0 + \Delta W$, where $W_0$ is the weight matrix of the pre-trained model and $\Delta W$ is the rank-$r$ approximation of the update matrix, i.e., the difference between the fully fine-tuned weights and the pre-trained weights.

Run the next cell to evaluate your model for different rank values. Start with the rank from the approximated model from the previous section and then halve the rank in each step. What is the lowest rank that still gives you a higher accuracy than the head-tuned model?

In [30]:
approximated_model_2 = make_approximated_model_2(768)
#approximated_model_2 = approximated_model_2.to("cuda")
is_on_cuda = next(approximated_model_2.parameters()).is_cuda
if is_on_cuda:
    print("GPUS")
else:
    print(":(")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


GPUS


In [42]:
RANK = 1
approximated_model_2 = make_approximated_model_2(RANK)
_headtuned_model = AutoModelForSequenceClassification.from_pretrained("headtuned").to("cuda")

lora_approx = evaluate(approximated_model_2)
ht_eval = evaluate(_headtuned_model)

print(f"LoRA-tuned.rank={RANK}: accuracy={lora_approx["eval_accuracy"]}")

print(f"Headtuned accuracy={ht_eval['eval_accuracy']}")

for rank in [1, 5, 10, 50, 100]:
    model = make_approximated_model_2(rank)
    print(f"Rank {rank}: accuracy={evaluate(model)['eval_accuracy']}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LoRA-tuned.rank=1: accuracy=0.856
Headtuned accuracy=0.812


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Rank 1: accuracy=0.856


...

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Rank 10: accuracy=0.878


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Rank 50: accuracy=0.882


Rank 100: accuracy=0.882


#### NOTE ANSWER:

- Lowest rank better than headtuned model is $r=1$ with  accuracy=$0.856$ compared to headtuned accuracy=$0.812$. 

## Low-Rank Adaptation (LoRA)

In this section, you will implement the LoRA adapters and fine-tune the adapted model.

### 🎓 Task 4.08: Implement the adapter

A LoRA adapter implements the forward function

$$
y = x W_0 + x \Delta W = x W_0 + x A B
$$

where $W_0$ is a linear transformation from the pre-trained model and $\Delta W$ is a learned update matrix, deconstructed into the product $AB$ of two rank-$r$ matrices $A$ and $B$. LoRA scales the update matrix $\Delta W$ by a factor of $\alpha / r$, where $\alpha$ is a hyperparameter. (To keep the formula tidy, we ignore the fact that the linear transformation in the pre-trained model may additionally include a bias.)

In [32]:
import torch.nn as nn


class LoRA(nn.Module):
    def __init__(self, pretrained: nn.Linear, rank: int = 12, alpha:int = 24):
        super().__init__()
        self.alpha: int = alpha
        self.rank: int = rank
        self.w: nn.Linear = pretrained
        self.w.weight.requires_grad = False
        self.w.bias.requires_grad = False
        self.A = nn.Parameter(torch.randn(pretrained.weight.shape[1], rank)) 
        self.B = nn.Parameter(torch.zeros(rank, pretrained.weight.shape[0]))
        

    def forward(self, x):
        pretrained_output = self.w.forward(x)
        lora_output = (self.alpha / self.rank) * x @ self.A @ self.B
        return pretrained_output + lora_output

Your code must comply with the following specification:

**__init__** (*self*, *pretrained*, *rank* = 12, *alpha* = 24)

> Initialises the LoRA adapter. This sets up the matrices $A$ and $B$ from the equation above. The matrix $A$ is initialised with random weights from a standard normal distribution; the matrix $B$ is initialised with zeros. The argument *pretrained* is the linear layer from the pre-trained model that should be adapted. The arguments *rank* and *alpha* are the rank $r$ and the hyperparameter $\alpha$ in the equation above.

**forward** (*self*, *x*)

> Sends an input *x* through the adapter, implementing the equation above.

### 🎓 Task 4.09: Inject the adapter into the pre-trained model

The final step is to construct an adapted model by injecting the LoRA adapters into the pre-trained model.

In [33]:
def make_lora_model(rank) -> nn.Module:
    _pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
    )
    for name, param in _pretrained_model.named_parameters():
        param.requires_grad = False
    for name, module in _pretrained_model.named_modules():
        if "attention.v_lin" in name or "attention.q_lin" in name and isinstance(module, nn.Linear):
            deep_setattr(_pretrained_model, name, LoRA(module, rank, alpha=2*rank))
    _pretrained_model = _pretrained_model.to("cuda")
    return train(_pretrained_model)

Implement the function to match the following specification:

> **make_lora_model** (*rank*)
>
> Returns a model that is identical to the pre-trained model, except that the query and value linear layers have been wrapped in LoRA adapters, and the LoRA adapters and the head layers of the pre-trained model have been trained on the sentiment data. (The other parameters of the pre-trained model are left untouched.) The rank of the adapters is specified by the argument *rank*. The *alpha* value of the adapters is set to twice the rank (a common rule of thumb).

Run the next cell to evaluate your model for $r = 6$ and $\alpha = 12$. How many trainable parameters does the adapted model have? What accuracy do you get? How do these value relate to the number of trainable parameters and accuracy of the fully fine-tuned model, in terms of percentages?

In [34]:
lora_model_rank6 = make_lora_model(6)
lora_model_rank12 = make_lora_model(12)
print(evaluate(lora_model_rank6))
print(evaluate(lora_model_rank12))

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Number of trainable parameters: 110592
Number of trainable parameters: 110592


Epoch,Training Loss,Validation Loss,Accuracy
1,0.459000,0.426023,0.828000
2,0.219800,0.337819,0.866000
3,0.203300,0.317230,0.876000


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Number of trainable parameters: 221184
Number of trainable parameters: 221184


Epoch,Training Loss,Validation Loss,Accuracy
1,0.417200,0.328120,0.862000
2,0.184900,0.319089,0.866000
3,0.171900,0.315445,0.864000


{'eval_loss': 0.31723007559776306, 'eval_model_preparation_time': 0.0008, 'eval_accuracy': 0.876, 'eval_runtime': 3.146, 'eval_samples_per_second': 158.931, 'eval_steps_per_second': 20.025}


{'eval_loss': 0.31544458866119385, 'eval_model_preparation_time': 0.0007, 'eval_accuracy': 0.864, 'eval_runtime': 3.1514, 'eval_samples_per_second': 158.66, 'eval_steps_per_second': 19.991}


### NOTE 4.09: ANSWER

| Model          | Rank | Alpha | Accuracy | % of Full Accuracy | Trainable Parameters | % of Pretrained | Training Time |
|----------------|------|-------|----------|-------------------|---------------------|-----------------|---------------|
| LoRA+pretrain  | 6    | 12    | 87.6%      | 99.1%             | 110,592             | 0.16%           | 01:43         |
| LoRA+pretrain  | 12   | 24    | 86.2%      | 97.3%             | 221,184             | 0.32%           | 01:47         |
| Full finetune  | N/A  | N/A   | 88.4%    | 100%              | ~66M                | 100%            | 02:12         |

The above table shows results from the LoRA experiments (row one and row two). With rank 6 we get great accuracy of 88%. With rank 6 we also only train a very small fraction of the wieghts, 0.16%. However, the training time did not see as drastic improvements. We only saw ~30 seconds of time improvements for both rank 6 and 12 compared to the full fine tune.

## Alternatives to Transformer-based models

Even with methods for parameter-efficient fine-tuning, applying DistilBERT and other Transformer-based models comes at a significant cost – an investment that does not always pay off. In the final task of this lab, we ask you to explore a more traditional approach to classification and contrast it with the pre-training/fine-tuning approach of neural language models.

### 🎓 Task 4.10: Comparing with a non-neural classifier

Browse the web to find a tutorial on how to apply a classifier from the [scikit-learn](https://scikit-learn.org/stable/) library to the problem of sentiment classification and implement the method here in this notebook. We suggest you use [multinomial Naive Bayes](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html) or [logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html). (Once you have code for one method, it is easy to switch to the other.) Evaluate your chosen classifier on the IMDB dataset.

Questions to consider:

* Which classifier did you try? What results did you get? How long did it take you to train and run the classifier?
* What is your perspective on the trade-off between accuracy and resource requirements between the two approaches?
* What did you learn? How, exactly, did you learn it? Why does this learning matter?

I tried the following classifiers:

- Logistic Regression
- Multinomial Naive Bayes

My results were 84.8% accuracy and 84.2% accuracy respectively for logistic regression and multinomial Naive Bayes. Both models and preprocessing took 0.25s compared to 1 minute and 47 seconds for the LoRA+pretrained model.

My perspective on the trade-off between using pretrain/fine-tuning and specific purpose-built language models is that it depends on two things: (1) the specifics of the downstream task, i.e., how important is it that we get high precision/recall for example? Is the model sensitive to data drift, needing significant infrastructure for re-tuning the model? (2) How interpretable does the model need to be? The pretrain/fine-tune paradigm lends itself poorly to scrutiny, especially with LLMs and proprietary models. In contrast, logistic regression and decision trees can easily be explained to stakeholders, whereas the inner workings of LLMs are open research questions.

Through this lab, I learned:

- **A lot about PyTorch:** Through (4.01) targeting specific layers for tuning using `param.requires_grad`, and (4.03, 4.04) extracting/replacing layers and their weights. Learning PyTorch matters to me because it unlocks how to work with modern machine learning literature and state-of-the-art language models.

- **An empirical study behind LoRA:** In 4.07, we showed that the performance of a model that used a low-rank decomposition $SVD(\Delta W) \approx \Delta W = W_{SFT} - W_0$ was close to the performance of a full fine-tune. This matters because it's easier to implement when I know what is going on mathematically.

- **Neural Networks Learning SVD:** I also learned that neural networks can "learn" decompositions if we set up the math/computational flow correctly, by simply stating the mathematics and letting SGD do its work. This matters because it gave me a deeper understanding of what happens in a neural network. Each update is a sum of gradients on top of the original weights.

- **Cost/Benefit Analysis/Reflection:** I also reflected on when to use pretrain/fine-tune models and when to consider using simpler models. Contrasting these two paradigms helped me contextualize and question the cost/benefit of LLMs. This matters because LLMs are expensive to develop/train/run and fine-tune. There are also ethical/environmental considerations that should be taken into account depending on the downstream task.

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
import time

t1 = time.time()

tfidf = TfidfVectorizer(max_features=10000)
X_train = tfidf.fit_transform(imdb_dataset["train"]["review"])
X_test = tfidf.transform(imdb_dataset["eval"]["review"])
y_train = imdb_dataset["train"]["label"]
y_test = imdb_dataset["eval"]["label"]
# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_accuracy = accuracy_score(y_test, lr.predict(X_test))
print(f"Logistic Regression: {lr_accuracy:.3f}")

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
nb_accuracy = accuracy_score(y_test, nb.predict(X_test))
print(f"Naive Bayes: {nb_accuracy:.3f}")
t2 = time.time()

print(f"Elapsed time: {t2-t1:.2f}s")

Logistic Regression: 0.848
Naive Bayes: 0.842
Elapsed time: 0.25s


**🥳 Congratulations on finishing this lab! 🥳**